# Illustrate the Enemy Formations

In [9]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:70% !important; }</style>"))

### Read in the formation data

In [12]:
"""
; The first six bytes select the movement strategy
; for the formation from enemyMovementStrategyLoPtrArray.
.BYTE $01 ; Movement Strategy: Enemy 1
.BYTE $01 ; Movement Strategy: Enemy 2
.BYTE $01 ; Movement Strategy: Enemy 3
.BYTE $01 ; Movement Strategy: Enemy 4
.BYTE $01 ; Movement Strategy: Enemy 5
.BYTE $00 ; Movement Strategy: Enemy 6
; The next six bytes select the inital Y position for
; each enemy.
.BYTE $9C ; Initial Y Position : Enemy 1
.BYTE $6C ; Initial Y Position : Enemy 2
.BYTE $B4 ; Initial Y Position : Enemy 3
.BYTE $84 ; Initial Y Position : Enemy 4
.BYTE $CC ; Initial Y Position : Enemy 5
.BYTE $00 ; Initial Y Position : Enemy 6
; THe delay before adding the next enemy, think of it as
; the spacing between enemies as they enter the screen.
.BYTE $05 ; Delay between spawning enemies.
.BYTE $00 ; Initial X Position of Enemy 1
; The last byte is the sprite to be used for the enemies.
.BYTE $09 ; Sprite Value for Enemies
.BYTE $00 ; End Sentinel
"""
game_data_file = "formation_data.asm"
lines_in_file = open(game_data_file,'r').readlines()

formations = []
strategies, y_positions = [],[]
for l in lines_in_file:
    if l.startswith("enemy") and strategies:
        formations += [(
            strategies,
            y_positions,
            delay,
            initial_x,
            sprite_value
        )]
        strategies, y_positions = [],[]
    if "BYTE" not in l:
        continue
    v = int(l[15:17],16)
    if v == 255: v = 2
    if "Movement Strategy" in l:
        strategies += [v]
    elif "Y Position" in l:
        y_positions += [v]
    elif "Delay" in l:
        delay = v
    elif "Initial X" in l:
        initial_x = v
    elif "Sprite Value" in l:
        sprite_value = "{:02X}".format(v)

formations += [(
    strategies,
    y_positions,
    delay,
    initial_x,
    sprite_value
)]
formations[21]

([38, 39, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], 8, 2, '04')

In [65]:
mkdir -p formations

In [26]:
from PIL import Image, ImageDraw, ImageFont
from itertools import pairwise

double_size = lambda img: img.resize((int(img.width * 2), int(img.height * 2)), Image.NEAREST)

for level in range(1,16):
    for i,formation in enumerate(formations):
        strategies, y_positions, delay, initial_x, sprite_value = formation
        sprite = Image.open(f"meanie_sprites/{level}_MEANIE_{sprite_value}.png")
        sprite = double_size(sprite)

        img = Image.new('RGBA', (500 , 500))
        x = 200
        py = 0
        pairs = list(zip(y_positions,strategies))
        for (y,s),(ny,ns) in pairwise(pairs):
            if not s: continue
            if not y: y = 100
            if not ny: ny = 100
            img.paste(sprite, (x*2,y*2))
            x -= delay * 2 if ny != y else int(sprite.width/2)
        img.crop((200,200,500,500)).save(f"formations/{level}_formation_{i}.png")
        #img.save(f"formations/{level}_formation_{i}.png")


In [27]:
0x9c,0x6c,0xb4,0x84,0xcc

(156, 108, 180, 132, 204)

### Formations per Level

In [28]:
level_formations_raw = """level1EnemyFormationOrder = $C590
        .BYTE $00,$19,$04,$0C,$01,$29,$02,$00
        .BYTE $FF
level2EnemyFormationOrder = $C599
        .BYTE $0C,$2F,$22,$08,$1E,$20,$02,$0C
        .BYTE $FF
level3EnemyFormationOrder = $C5A2
        .BYTE $21,$06,$1F,$1C,$25,$0F,$31,$26
        .BYTE $21,$FF
level4EnemyFormationOrder = $C5AC
        .BYTE $1D,$07,$1A,$0A,$34,$0B,$0D,$31
        .BYTE $1D,$FF
level5EnemyFormationOrder = $C5B6
        .BYTE $04,$0E,$2A,$1C,$37,$32,$23,$17
        .BYTE $2E,$04,$FF
level6EnemyFormationOrder = $C5C1
        .BYTE $29,$1E,$0D,$2C,$0F,$1F,$2D,$2F
        .BYTE $0A,$29,$FF
level7EnemyFormationOrder = $C5CC
        .BYTE $0B,$05,$1C,$0C,$16,$2E,$36,$11
        .BYTE $02,$0A,$0B,$FF
level8EnemyFormationOrder = $C5D8
        .BYTE $07,$19,$03,$17,$24,$1D,$02,$21
        .BYTE $0E,$0D,$07,$FF
level9EnemyFormationOrder = $C5E4
        .BYTE $18,$09,$11,$30,$0A,$35,$0D,$26
        .BYTE $2B,$23,$17,$18,$FF
level10EnemyFormationOrder = $C5F1
        .BYTE $2F,$1B,$11,$25,$2A,$33,$31
        .BYTE $08,$1C,$10,$06,$2F,$FF
level11EnemyFormationOrder = $C5FE
        .BYTE $05,$16,$35,$27,$0D,$22,$0A,$00
        .BYTE $36,$1D,$2F,$19,$05,$FF
level12EnemyFormationOrder = $C60C
        .BYTE $37,$1C,$08,$1E,$2F,$2C,$28,$20
        .BYTE $34,$16,$2D,$1F,$37,$FF
level13EnemyFormationOrder = $C61A
        .BYTE $35,$18,$33,$09,$0B,$2A,$00,$0E
        .BYTE $31,$16,$2C,$29,$37,$35,$FF
level14EnemyFormationOrder = $C629
        .BYTE $36,$2E,$0D,$16,$1B,$1A,$1D,$04
        .BYTE $20,$28,$30,$27,$03,$36,$FF
level15EnemyFormationOrder = $C638
        .BYTE $38,$19,$34,$31,$20,$06,$18,$32
        .BYTE $30,$16,$16,$1F,$0C,$35,$38,$FF
"""

In [39]:
level_formations_list = level_formations_raw.split('\n')[0:-1]
level_formations_list
level_formations = []
for i in range(0,len(level_formations_list),3):
    bs = level_formations_list[i+1][14:].split(',')
    bs += level_formations_list[i+2][14:].split(',')
    ns = [int(x[1:],16) for x in bs]
    level_formations += [ns]

In [40]:
mkdir -p formations/level_formations

In [62]:
from PIL import Image, ImageDraw, ImageFont
for l,formations in enumerate(level_formations):
    level = l+1
    img = Image.new('RGBA', (2400, 850 if len(formations) > 9 else 450))
    draw = ImageDraw.Draw(img)
    x,y = 0,0
    for f in formations:
        if f == 255:
            break
        formation = Image.open(f"formations/{level}_formation_{f}.png")
        img.paste(formation, (x,y))

        label_text = ("0"+str(f))[-2:]
        label_fnt = ImageFont.truetype("Eurostile.ttf", 52)
        label_x = x + 110
        label_y = y + 280
        draw.text((label_x, label_y), label_text, font=label_fnt, fill="black")

        label_text = "$"+ '{:02X}'.format(f)
        label_fnt = ImageFont.truetype("JetBrainsMono-Regular.ttf", 62)
        label_x = x + 100
        label_y = y + 320
        draw.text((label_x, label_y), label_text, font=label_fnt, fill="black")
        
        x+= 300
        if x > 2100:
            y+=450
            x=0
    img.save(f"formations/level_formations/{level}_formations.png")


In [ ]:
from PIL import Image, ImageDraw, ImageFont

for level in range(1,16):
    fore_img = Image.open(f"surfaces_raw/{level}_full.png")
    img = Image.new('RGBA', (fore_img.width , 30 + fore_img.height + 180))
    img.paste(fore_img, (0, 30))

    draw = ImageDraw.Draw(img)
    SEGMENT_WIDTH = 32 * 8
    clip_start = 0
    clip_end = SEGMENT_WIDTH
    ADDR = 0x8200
    for i in range(0,16):
        draw.rectangle([(clip_start,0),(clip_start,img.height-1)], 
                       fill=None, outline=(0,0,0,255), width=2)

        label_text = ("0"+str(i))[-2:]
        label_fnt = ImageFont.truetype("Eurostile.ttf", 52)
        label_x = int(clip_start + ((clip_end - clip_start) / 2) + -40)
        label_y = 30 + fore_img.height + 30
        draw.text((label_x, label_y), label_text, font=label_fnt, fill="black")

        label_text = "$"+hex(ADDR)[2:].upper()
        label_fnt = ImageFont.truetype("JetBrainsMono-Regular.ttf", 62)
        label_x = int(clip_start + ((clip_end - clip_start) / 2) + -90)
        label_y = 90 + fore_img.height + 30
        draw.text((label_x, label_y), label_text, font=label_fnt, fill="black")
        ADDR += 0x20
        
        clip_start += SEGMENT_WIDTH
        clip_end += SEGMENT_WIDTH
    
    img.save(f"scrolling/segments/{level}_scroll_segments_addresses.png")

img